In [12]:
import sys
from pathlib import Path

try:
    ROOT = Path(globals()['__vsc_ipynb_file__']).resolve().parent.parent
except KeyError:
    ROOT = Path.cwd()
    for _ in range(4):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.configs.text_configs import get_roberta_afc_config, get_roberta_afd_config
from src.experiments.mmused_text import (
    DEFAULT_DATA_PATH,
    DEFAULT_RESULTS_PATH,
    make_mmused_fallacy_loader,
    prepare_text_reproducibility,
    run_mmused_text_cv,
)
from src.utils.results import ResultsManager

DATA_PATH = DEFAULT_DATA_PATH
RESULTS_PATH = DEFAULT_RESULTS_PATH


In [13]:
# Seeds + HF cache + matmul precision (call once per kernel restart)
prepare_text_reproducibility(seed=42)


Seed set to 42


# NOTE: We use MAMKit for:
- Data loading: MMUSEDFallacy, get_splits('mancini-et-al-2024')
- Model: mamkit.models.text.Transformer
- Collators: TextTransformerCollator, UnimodalCollator
- Training loop: MAMKitLightingModel

The MAMKit config 'mancini-et-al-2024' has a known bug:
 TypeError: BaseConfig.__init__() missing required argument: 'seeds'
We define hyperparameters manually following the paper architecture,
with class weights computed from our data distribution.


In [14]:
# TRANSFORMERS_CACHE is set in prepare_text_reproducibility() (cell above).


### Reproducible run (project code in `src/experiments`)

All of this still uses **MAMKit** (`MMUSEDFallacy`, splits, models via `TextTrainer`). Run **loaders** (next cell), then **AFC** (35-fold CV + BM3), then **AFD** (same CV layout; **much slower** — ~17k sentences).


In [15]:
# Build loaders once; training cells reuse `loader_afc` / `loader_afd`.
loader_afc = make_mmused_fallacy_loader("afc")
loader_afd = make_mmused_fallacy_loader("afd")

print(f"AFC: {loader_afc.data.shape}")
print(f"AFD: {loader_afd.data.shape}")


Building AFC Context: 100%|██████████| 1278/1278 [00:00<00:00, 17052.46it/s]


AFC: (1278, 8)


Building AFD data...: 100%|██████████| 35/35 [00:01<00:00, 33.13it/s]


AFD: (17118, 7)


### Run AFC offline (survives SSH / editor disconnect)

The **notebook kernel** can stop if your remote session drops. For a long CV + BM3, use the **same logic** from a terminal so it keeps running:

```bash
cd /mm_argfallacy
mkdir -p logs
nohup ./mmarg_env/bin/python scripts/run_roberta_afc.py > logs/roberta_afc.log 2>&1 &
tail -f logs/roberta_afc.log   # optional: watch progress; Ctrl+C stops tail only
```

**tmux** (reattach after reconnect: `tmux attach -t roberta_afc`): open a session, run `./mmarg_env/bin/python scripts/run_roberta_afc.py`, then detach with **Ctrl+B** then **D**.

Progress is still written to `results/results.json` and checkpoints under `checkpoints/` as usual.


In [ ]:
# AFC — full leave-one-dialogue-out CV (35 folds) + BM3 refit (3 checkpoints on disk).
summary_afc = run_mmused_text_cv(
    get_roberta_afc_config(),
    loader=loader_afc,
    save_bm3_checkpoints_after=True,
)
print(summary_afc)

# Smoke test (uncomment):
# summary_afc = run_mmused_text_cv(
#     get_roberta_afc_config(),
#     loader=loader_afc,
#     max_folds=3,
#     save_bm3_checkpoints_after=False,
# )


### AFD — Argument Fallacy Detection (binary, sentence-level)

Same CV protocol (**35 folds**, BM3 checkpoints after full CV). Dataset is **~17k sentences** — expect a **long** GPU run; use **`scripts/run_roberta_afd.py`** with `nohup` or **tmux** like AFC.


In [ ]:
# AFD — full 35-fold CV + BM3 (binary F1). Very slow compared to AFC.
summary_afd = run_mmused_text_cv(
    get_roberta_afd_config(),
    loader=loader_afd,
    save_bm3_checkpoints_after=True,
)
print(summary_afd)

# Smoke test (uncomment):
# summary_afd = run_mmused_text_cv(
#     get_roberta_afd_config(),
#     loader=loader_afd,
#     max_folds=2,
#     save_bm3_checkpoints_after=False,
# )


### Run AFD offline (survives SSH disconnect)

```bash
cd /mm_argfallacy
mkdir -p logs
nohup ./mmarg_env/bin/python scripts/run_roberta_afd.py > logs/roberta_afd.log 2>&1 &
tail -f logs/roberta_afd.log
```


## AFC & AFD — results and interpretation

After running both tasks, `ResultsManager.print_comparison_table()` lists every experiment in `results/results.json` (macro F1 for AFC, binary F1 for AFD).


In [8]:
rm = ResultsManager(str(RESULTS_PATH))
rm.print_comparison_table()



Experiment                Task   Metric              Mean      Std  Folds
roberta_afc               afc    test_macro_f1     0.4489   0.1922     35

Paper baselines
roberta_afc (paper)       afc    macro_f1          0.3925
roberta_afd (paper)       afd    binary_f1         0.2770
